## Dataset Export & Documentation

### By:
MGO

### Date:
2026-03-27

### Description:

Export clean, documented datasets ready for research use. This is what a CIVICUS
"documented dataset" deliverable looks like: structured data with a comprehensive
data dictionary, methodology notes, and reproducibility instructions.

**Output:** `data/06_reporting/` (CSV + Parquet exports with documentation)

## Import libraries

In [ ]:
from __future__ import annotations

import json
import sys
from datetime import datetime
from pathlib import Path

import duckdb
import pandas as pd

# Add src to path for importing project utilities
sys.path.insert(0, str(Path("../../src").resolve()))

## Load enriched data

Load from `data/04_enriched/` (post-NLP analysis). If enriched data is not yet
available, fall back to intermediate or sample data.

In [ ]:
data_dir = Path("../../data")
enriched_dir = data_dir / "04_enriched"
intermediate_dir = data_dir / "02_intermediate"
sample_path = data_dir / "sample" / "sample_posts.json"

# Try enriched data first, then intermediate, then sample
enriched_path = enriched_dir / "posts_with_sentiment.parquet"
intermediate_path = intermediate_dir / "all_posts_cleaned.parquet"

if enriched_path.exists():
    df = pd.read_parquet(enriched_path)
    data_source = "enriched (post-NLP)"
elif intermediate_path.exists():
    df = pd.read_parquet(intermediate_path)
    data_source = "intermediate (pre-NLP)"
elif sample_path.exists():
    with open(sample_path) as f:
        df = pd.DataFrame(json.load(f))
    data_source = "sample (demonstration)"
else:
    df = pd.DataFrame()
    data_source = "none"

print(f"Data source: {data_source}")
print(f"Shape: {df.shape}")
if not df.empty:
    print(f"Columns: {df.columns.tolist()}")
    if "timestamp" in df.columns:
        df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
        print(f"Date range: {df['timestamp'].min()} to {df['timestamp'].max()}")

## Data Dictionary

Comprehensive data dictionary for the exported dataset. This documents every column
so that downstream researchers can understand what each field contains.

In [ ]:
# Define the data dictionary
data_dictionary = pd.DataFrame([
    {
        "Column": "id",
        "Type": "str",
        "Description": "Unique post identifier (SHA-256 hash of content)",
        "Example": "a1b2c3d4e5f67890",
        "Source": "Generated",
    },
    {
        "Column": "text",
        "Type": "str",
        "Description": "Original post text content",
        "Example": "Petro anuncia reforma pensional...",
        "Source": "RSS/Reddit/GDELT",
    },
    {
        "Column": "source",
        "Type": "str",
        "Description": "Specific collection source name",
        "Example": "El Tiempo RSS",
        "Source": "Collector",
    },
    {
        "Column": "platform",
        "Type": "str",
        "Description": "Platform type (rss, reddit, gdelt, acled)",
        "Example": "rss",
        "Source": "Collector",
    },
    {
        "Column": "author",
        "Type": "str",
        "Description": "Post author or source attribution",
        "Example": "username123",
        "Source": "Source metadata",
    },
    {
        "Column": "timestamp",
        "Type": "datetime",
        "Description": "Publication timestamp",
        "Example": "2026-03-22T14:30:00",
        "Source": "Source metadata",
    },
    {
        "Column": "url",
        "Type": "str",
        "Description": "Link to original content",
        "Example": "https://eltiempo.com/...",
        "Source": "Source metadata",
    },
    {
        "Column": "sentiment_label",
        "Type": "str",
        "Description": "Sentiment classification (POS/NEG/NEU)",
        "Example": "NEG",
        "Source": "pysentimiento",
    },
    {
        "Column": "prob_pos",
        "Type": "float",
        "Description": "Positive sentiment probability",
        "Example": "0.12",
        "Source": "pysentimiento",
    },
    {
        "Column": "prob_neg",
        "Type": "float",
        "Description": "Negative sentiment probability",
        "Example": "0.87",
        "Source": "pysentimiento",
    },
    {
        "Column": "prob_neu",
        "Type": "float",
        "Description": "Neutral sentiment probability",
        "Example": "0.01",
        "Source": "pysentimiento",
    },
    {
        "Column": "entities",
        "Type": "list[str]",
        "Description": "Named entities extracted from text",
        "Example": '["Petro", "Bogota"]',
        "Source": "spaCy es_core_news_lg",
    },
    {
        "Column": "topic_id",
        "Type": "int",
        "Description": "Topic cluster assignment",
        "Example": "3",
        "Source": "sentence-transformers + sklearn",
    },
    {
        "Column": "topic_label",
        "Type": "str",
        "Description": "Topic cluster description",
        "Example": "reforma pensional",
        "Source": "sentence-transformers + sklearn",
    },
])

print("Data Dictionary:")
display(
    data_dictionary.style.set_properties(**{"text-align": "left"})
    .set_table_styles([{"selector": "th", "props": [("text-align", "left")]}])
)

## Export Datasets

Export all datasets in both CSV (human-readable, universal) and Parquet (efficient,
typed) formats to `data/06_reporting/`.

In [ ]:
reporting_dir = data_dir / "06_reporting"
reporting_dir.mkdir(parents=True, exist_ok=True)

exported_files = []

if not df.empty:
    # Export main dataset as CSV + Parquet
    csv_path = reporting_dir / "pulso_electoral_main_dataset.csv"
    parquet_path = reporting_dir / "pulso_electoral_main_dataset.parquet"
    df.to_csv(csv_path, index=False, encoding="utf-8")
    df.to_parquet(parquet_path, index=False)
    exported_files.append({"file": csv_path.name, "format": "CSV", "rows": len(df)})
    exported_files.append({"file": parquet_path.name, "format": "Parquet", "rows": len(df)})
    print(f"Exported main dataset: {len(df)} rows")

# Export ACLED events if available
acled_raw = data_dir / "01_raw" / "acled"
if acled_raw.exists():
    acled_files = sorted(acled_raw.glob("*.json"))
    if acled_files:
        acled_data = []
        for f in acled_files:
            with open(f) as fh:
                acled_data.extend(json.load(fh))
        if acled_data:
            df_acled = pd.DataFrame(acled_data)
            acled_csv = reporting_dir / "acled_events_colombia.csv"
            df_acled.to_csv(acled_csv, index=False, encoding="utf-8")
            exported_files.append({"file": acled_csv.name, "format": "CSV", "rows": len(df_acled)})
            print(f"Exported ACLED events: {len(df_acled)} rows")

# Export GDELT events if available
gdelt_raw = data_dir / "01_raw" / "gdelt"
if gdelt_raw.exists():
    gdelt_files = sorted(gdelt_raw.glob("*.json"))
    if gdelt_files:
        gdelt_data = []
        for f in gdelt_files:
            with open(f) as fh:
                gdelt_data.extend(json.load(fh))
        if gdelt_data:
            df_gdelt = pd.DataFrame(gdelt_data)
            gdelt_csv = reporting_dir / "gdelt_articles_colombia.csv"
            df_gdelt.to_csv(gdelt_csv, index=False, encoding="utf-8")
            exported_files.append({"file": gdelt_csv.name, "format": "CSV", "rows": len(df_gdelt)})
            print(f"Exported GDELT articles: {len(df_gdelt)} rows")

# Export topic model results if available
topic_path = data_dir / "05_analysis" / "topic_model_results.parquet"
if topic_path.exists():
    df_topics = pd.read_parquet(topic_path)
    topics_csv = reporting_dir / "topic_model_results.csv"
    df_topics.to_csv(topics_csv, index=False, encoding="utf-8")
    exported_files.append({"file": topics_csv.name, "format": "CSV", "rows": len(df_topics)})
    print(f"Exported topic model results: {len(df_topics)} rows")

print(f"\nTotal files exported: {len(exported_files)}")

## Dataset Statistics Summary

In [ ]:
if not df.empty:
    print("=" * 60)
    print("DATASET STATISTICS SUMMARY")
    print("=" * 60)
    print(f"\nTotal records: {len(df)}")

    # Date coverage
    if "timestamp" in df.columns:
        print(f"Date range: {df['timestamp'].min()} to {df['timestamp'].max()}")

    # Platform distribution
    if "platform" in df.columns:
        print(f"\nPlatform distribution:")
        for platform, count in df["platform"].value_counts().items():
            pct = count / len(df) * 100
            print(f"  {platform}: {count} ({pct:.1f}%)")

    # Sentiment distribution (if available)
    if "sentiment_label" in df.columns:
        print(f"\nSentiment distribution:")
        for label, count in df["sentiment_label"].value_counts().items():
            pct = count / len(df) * 100
            print(f"  {label}: {count} ({pct:.1f}%)")
else:
    print("No data available for statistics")

## Methodology Note (Template)

Auto-generated methodology note documenting how this dataset was created.

In [ ]:
methodology = f"""
# Methodology Note -- Pulso Electoral Colombia 2026

## Data Sources
- **RSS Feeds**: {', '.join(['El Tiempo', 'El Espectador', 'Semana', 'La Silla Vacia',
                             'Caracol Radio', 'Blu Radio', 'Razon Publica', 'Pulzo'])}
- **Reddit**: r/Colombia political discussions via PRAW API
- **GDELT**: Global Database of Events, Language and Tone (DOC API)
- **ACLED**: Armed Conflict Location and Event Data Project

## Collection Period
- Start: 2026-03-22
- End: {datetime.now().strftime('%Y-%m-%d')}
- Total records: {len(df) if not df.empty else 'N/A'}

## Filtering Criteria
- Geographic focus: Colombia
- Language: Spanish (filtered with langdetect)
- Topic: Political discourse, elections, civic space
- Minimum text length: 10 characters
- Exact duplicates removed

## NLP Models Applied
- **Sentiment analysis**: pysentimiento/robertuito-sentiment-analysis
  (HuggingFace, trained on ~500M Spanish tweets)
- **Named Entity Recognition**: spaCy es_core_news_lg v3.8+
- **Topic modeling**: sentence-transformers (paraphrase-multilingual-MiniLM-L12-v2) + sklearn clustering

- **Emotion analysis**: pysentimiento/robertuito-emotion-analysis

## Known Limitations
- RSS feeds may miss articles published only on social media
- Reddit represents a specific demographic (younger, urban, tech-savvy)
- ACLED has reporting lag of days to weeks
- Sentiment models may misclassify Colombian slang or sarcasm
- Topic model quality depends on corpus size

## Ethical Considerations
- Only publicly available data was collected
- No personal messages or private group content
- Reddit usernames are pseudonymous
- Data is used for research purposes only
- Complies with CIVICUS research ethics guidelines

## Reproducibility
- All collection code is in `notebooks/1-data/`
- All analysis code is in `notebooks/3-analysis/`
- Configuration files in `conf/`
- Environment: Python 3.12, dependencies in pyproject.toml
"""

# Save methodology note
methodology_path = reporting_dir / "METHODOLOGY.md"
with open(methodology_path, "w", encoding="utf-8") as f:
    f.write(methodology)
print(f"Saved methodology note to {methodology_path}")
print("\n" + methodology)

## Reproducibility

Exact commands to recreate this dataset from scratch.

In [ ]:
reproducibility_steps = """
# Reproduce this dataset from scratch:

# 1. Set up environment
make install_env
cp .env.example .env
# Edit .env with your API credentials (ACLED, Reddit)
make download_models

# 2. Collect data
make collect_rss              # RSS feeds (no credentials needed)
make collect_reddit           # Reddit (needs REDDIT_CLIENT_ID, etc.)
# Notebook 03: GDELT + ACLED (GDELT: no auth; ACLED: needs ACLED_EMAIL + ACLED_PASSWORD)
uv run jupyter execute notebooks/1-data/03_mgo_gdelt_acled_collection_20260323.ipynb

# 3. Explore data
uv run jupyter execute notebooks/2-exploration/01_mgo_data_overview_20260324.ipynb

# 4. Run analysis
make analyze

# 5. Export datasets
uv run jupyter execute notebooks/4-output/01_mgo_dataset_export_20260327.ipynb

# Configuration files used:
#   conf/data_collection/rss_feeds.yml
#   conf/data_collection/reddit.yml
#   conf/data_collection/gdelt.yml
#   conf/data_collection/acled.yml
#   conf/keywords/election.yml
#   conf/keywords/manipulation.yml
#   conf/keywords/civic_space.yml
#   conf/nlp/models.yml

# Model versions:
#   pysentimiento: 0.7.x (robertuito-sentiment-analysis)
#   spaCy: 3.8.x (es_core_news_lg)
#   (BERTopic removed -- using sentence-transformers + sklearn directly)
#   sentence-transformers: 3.4.x (paraphrase-multilingual-MiniLM-L12-v2)
"""
print(reproducibility_steps)

In [ ]:
# Generate manifest of all exported files
manifest = {
    "project": "Pulso Electoral Colombia 2026",
    "generated_at": datetime.now().isoformat(),
    "description": "Documented dataset export for CIVICUS DDI research",
    "files": exported_files,
}

manifest_path = reporting_dir / "manifest.json"
with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2, default=str)

print(f"Saved manifest to {manifest_path}")
print(json.dumps(manifest, indent=2, default=str))

## References

- CIVICUS Digital Democracy Initiative: https://civicus.org
- pysentimiento: https://github.com/pysentimiento/pysentimiento
- spaCy: https://spacy.io/
- sentence-transformers: https://www.sbert.net/
- DuckDB: https://duckdb.org/